# EPA UIC aquifer exemptions — Texas overview

Visualize EPA Underground Injection Control (UIC) aquifer exemptions in Texas, ranked by injection-zone depth. Useful as a quick audit of where the groundwater-protection exclusions are concentrated.

In [ ]:
%pip install --quiet 'sondio[geo]>=0.1.2,<0.2' matplotlib

# sondio >= 0.1.2 auto-resolves your key from (in order):
#   1. Colab Secrets — add a secret named SONDIO_API_KEY (🔑 sidebar, toggle Notebook access)
#   2. Kaggle Secrets — add a secret named SONDIO_API_KEY (Add-ons → Secrets)
#   3. SONDIO_API_KEY environment variable (Deepnote / Hex / SageMaker / local)
#   4. ~/.sondio/config (local CLI users)
# Grab a free key at https://sondio.io/developers
import sondio
print(f"sondio {sondio.__version__}")

In [ ]:
ae = sondio.us.epa.aquifer_exemptions(state="TX", all_pages=True)
print(f"{len(ae)} exemptions")
ae.head()

In [ ]:
# Shallowest exemptions are the most scrutinized — a few hundred feet.
shallow = ae.dropna(subset=["depth_ft"]).sort_values("depth_ft").head(20)
shallow[["well_class", "county", "injection_zone", "depth_ft", "latitude", "longitude"]]

In [ ]:
# County rollup — where do the exemptions cluster?
by_county = (
    ae.dropna(subset=["county"])
      .groupby("county").size().sort_values(ascending=False).head(15)
)
by_county

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
pts = ae.dropna(subset=["latitude", "longitude"])
ax.scatter(pts["longitude"], pts["latitude"], s=6, c=pts["depth_ft"], cmap="viridis", alpha=0.7)
ax.set_title("Texas UIC aquifer exemptions (color = depth_ft)")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
plt.show()

## Next steps
- Overlay with `sondio.geo.subdivisions(country="US", with_geometry=True)` for a true choropleth.
- Cross-reference with `sondio.us.epa.impaired_waters(state="TX")` to flag collocations.
- Add `well_class="II"` to isolate oil & gas UIC wells specifically.